In [1]:
import os
from tqdm import tqdm
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.font_manager as font_manager
font_manager.fontManager.addfont("/home/tx82/Arial.ttf")
plt.rcParams['font.family'] = 'Arial'
import matplotlib.ticker as mticker
from scipy.stats import mannwhitneyu
from itertools import combinations

In [2]:
root_dir = "/fs/cbsuhy02/storage/tx82/"

In [3]:
# load gene table (protein coding genes, using gencode v46 annotation)
gene_table_file = os.path.join(root_dir, "_DATA/_COMMON/_reference/gencode/gencode.v46.annotation.coding.tsv")
gene_table = pd.read_table(gene_table_file)
print("Number of all protein coding genes:", len(gene_table["gene_name"].unique()))

# load metadata and id-mapping (UniProt to gene name)
df_annot = pd.read_csv(os.path.join(root_dir, "_DATA/_COMMON/_reference/HUMAN_9606_idmapping.dat.gz"), sep = "\t", dtype = str, header = None)
df_annot[0] = df_annot[0].apply(lambda x: x.split("-")[0])
df_annot[2] = df_annot[2].apply(lambda x: x.split(".")[0])
df_annot = df_annot[[0, 2]].drop_duplicates().rename(columns = {0: "UniProt"})

# filter the id_mapping file by gene names from the gene table
id_mapping = df_annot[df_annot[2].isin(list(gene_table["gene_name"]))]
id_mapping = id_mapping.rename(columns={2: "gene_name"})
id_mapping.head()

Number of all protein coding genes: 20036


,UniProt,gene_name
1,P31946,YWHAB
118,P62258,YWHAE
246,Q04917,YWHAH
336,P61981,YWHAG
462,P31947,SFN


In [ ]:
## Merge and process HINT lists

def fill_gene_name(gene_orig, gene_mapped): 
    # if there is a gene_name in the HINT, then use the original gene_name, otherwise mapped with the gene_name in the idmapping
    # !! but the original gene names in HINT might not be the approved symbols 
    if pd.isna(gene_orig) == False:
        return gene_orig
    else:
        return gene_mapped
    
# concatenate interactomes (downloaded from HINT)
HINT_file_list = ["HomoSapiens_cocomp_all.txt", "HomoSapiens_binary_all.txt"]
HINT_interactomes = []
for HINT_file in HINT_file_list:
    HINT_file_path = os.path.join(root_dir, "_DATA/_COMMON/network/HINT/", HINT_file)
    interactome = pd.read_table(HINT_file_path)
    tqdm.pandas()
    interactome["Uniprot_A"] = interactome["Uniprot_A"].apply(lambda x: x.split("-")[0]) # ignore isoforms
    interactome["Uniprot_B"] = interactome["Uniprot_B"].apply(lambda x: x.split("-")[0]) # ignore isoforms
    interactome = interactome[interactome["Uniprot_A"] != interactome["Uniprot_B"]] # remove self_loop
    interactome["PPI_pair"] = interactome.progress_apply(lambda row: ":".join(sorted([row["Uniprot_A"],row["Uniprot_B"]])), axis=1) # sorted PPI pairs
    interactome = interactome.drop_duplicates(subset=["PPI_pair"])
    # display(interactome.head())
    HINT_interactomes.append(interactome)
HINT_all_interactome = pd.concat(HINT_interactomes, axis=0)
HINT_all_interactome = HINT_all_interactome.drop_duplicates(subset=["PPI_pair"])
HINT_all_interactome = HINT_all_interactome[["Uniprot_A", "Uniprot_B", "PPI_pair", "Gene_A", "Gene_B"]]

# map uniprot IDs to gene names using id-mapping, and fill gene names (if NA in HINT then use mapped gene names, if have the original names in HINT then use the riginal names)
HINT_all_interactome = pd.merge(HINT_all_interactome, id_mapping, left_on="Uniprot_A", right_on="UniProt", how="left").rename(columns={"gene_name": "Gene_name_A"}).drop(columns=["UniProt"])
HINT_all_interactome = pd.merge(HINT_all_interactome, id_mapping, left_on="Uniprot_B", right_on="UniProt", how="left").rename(columns={"gene_name": "Gene_name_B"}).drop(columns=["UniProt"])
HINT_all_interactome["Gene_filled_A"] = HINT_all_interactome.progress_apply(lambda row: fill_gene_name(row["Gene_A"], row["Gene_name_A"]), axis=1)
HINT_all_interactome["Gene_filled_B"] = HINT_all_interactome.progress_apply(lambda row: fill_gene_name(row["Gene_B"], row["Gene_name_B"]), axis=1)

# use the mapped and clean gene names to generate gene pairs
HINT_all_interactome = HINT_all_interactome[["Uniprot_A", "Uniprot_B", "PPI_pair", "Gene_filled_A", "Gene_filled_B"]]
HINT_all_interactome = HINT_all_interactome.dropna(how="any")
HINT_all_interactome["gene_pair"] = HINT_all_interactome.apply(lambda row: ":".join(sorted([row["Gene_filled_A"], row["Gene_filled_B"]])), axis=1)
HINT_all_interactome = HINT_all_interactome.drop_duplicates(subset=["gene_pair"]).reset_index(drop=True)

# !! Issue: but many original gene names in HINT are not the approved HGNC symbols, need to map them to the approved symbols

# # generate the gene list for those genes that are not approved HGNC symbols
# hgnc_gene_info = pd.read_table(os.path.join(root_dir, "_DATA/_COMMON/genes/HGNC_symbols/HGNC_proteincoding.txt"))
# HINT_all_interactome_genes = pd.DataFrame(list(set(list(HINT_all_interactome["Gene_filled_A"].unique()) + list(HINT_all_interactome["Gene_filled_B"].unique()))))
# HINT_all_interactome_genes[~HINT_all_interactome_genes[0].isin(list(hgnc_gene_info["symbol"]))].to_csv(os.path.join(root_dir, "_DATA/_COMMON/network/HINT/HomoSapiens_both_all_genes_notapproved.txt"), sep='\t', index=None, header=None)

# load the appoved HGNC mapping only for those genes that are not approved symbols (converted from the HGNC website, see the file "HomoSapiens_both_all_genes_notapproved_hgnc.csv")
HINT_all_notapproved_baits_genes_hgnc = pd.read_table(os.path.join(root_dir, "_DATA/_COMMON/network/HINT/HomoSapiens_both_all_genes_notapproved_hgnc.csv"), sep=',')
genes_approved = []
for input_gene in HINT_all_notapproved_baits_genes_hgnc["Input"].unique():
    hgnc_inputgene = HINT_all_notapproved_baits_genes_hgnc[HINT_all_notapproved_baits_genes_hgnc["Input"] == input_gene]
    hgnc_inputgene_appr = hgnc_inputgene[hgnc_inputgene["Match type"] == "Approved symbol"]
    if len(hgnc_inputgene_appr) == 0: # for input genes that are not approved HGNC symbols, replace with all the mapped approved symbols
        print("Not the approved symbol:", input_gene)
        # display(hgnc_inputgene)
        genes_approved.append(hgnc_inputgene[["Input", "Approved symbol"]])
    elif len(hgnc_inputgene_appr) == 1:
        genes_approved.append(hgnc_inputgene_appr[["Input", "Approved symbol"]])
    elif len(hgnc_inputgene_appr) > 1:
        # display(genes_hgnc_inputgene_apr)
        genes_approved.append(hgnc_inputgene_appr[["Input", "Approved symbol"]])
genes_approved = pd.concat(genes_approved, axis=0).reset_index(drop=True)
display(genes_approved.head())

# finally, using the clean, filled and mostly approved symbols as gene names to generate gene_pairs and filter out duplicated ones
HINT_all_interactome = pd.merge(HINT_all_interactome, genes_approved, left_on="Gene_filled_A", right_on="Input", how="left").rename(columns={"Approved symbol":"Gene_approved_A"}).drop(columns=["Input"])
HINT_all_interactome["Gene_approved_A"] = HINT_all_interactome.apply(lambda row: row["Gene_approved_A"] if pd.isna(row["Gene_approved_A"]) == False else row["Gene_filled_A"], axis=1) # if no approved symbol, just use the original one
HINT_all_interactome = pd.merge(HINT_all_interactome, genes_approved, left_on="Gene_filled_B", right_on="Input", how="left").rename(columns={"Approved symbol":"Gene_approved_B"}).drop(columns=["Input"])
HINT_all_interactome["Gene_approved_B"] = HINT_all_interactome.apply(lambda row: row["Gene_approved_B"] if pd.isna(row["Gene_approved_B"]) == False else row["Gene_filled_B"], axis=1) # if no approved symbol, just use the original one
HINT_all_interactome["gene_pair"] = HINT_all_interactome.apply(lambda row: ":".join(sorted([row["Gene_approved_A"], row["Gene_approved_B"]])), axis=1)
HINT_all_interactome = HINT_all_interactome.drop_duplicates(subset=["gene_pair"]).reset_index(drop=True)
HINT_all_interactome = HINT_all_interactome[["Uniprot_A", "Uniprot_B", "PPI_pair", "Gene_approved_A", "Gene_approved_B", "gene_pair"]]

# save the merged interactome with filled gene names and approved symbols, and the gene pairs with approved symbols
HINT_all_interactome_file = os.path.join(root_dir, "_DATA/_COMMON/network/HINT/HomoSapiens_both_all.txt")
HINT_all_interactome.to_csv(HINT_all_interactome_file, sep='\t', index=None)
print("Number of interactions:", len(HINT_all_interactome))
HINT_all_interactome.head()

100%|██████████| 851183/851183 [00:04<00:00, 177953.80it/s]


Not the approved symbol: LOC554223
Not the approved symbol: CEFIP
Not the approved symbol: MUXA
Not the approved symbol: HEL-S-123m
Not the approved symbol: C16orf35
Not the approved symbol: RAB7L1
Not the approved symbol: BPGF-1
Not the approved symbol: NR1A2
Not the approved symbol: HEL-S-72p
Not the approved symbol: HEL-S-97n
Not the approved symbol: LOC51064
Not the approved symbol: C1orf19
Not the approved symbol: URCC5
Not the approved symbol: YARS
Not the approved symbol: IL-6SAG
Not the approved symbol: G7
Not the approved symbol: ARHH
Not the approved symbol: L27a
Not the approved symbol: SEPT3
Not the approved symbol: TC4
Not the approved symbol: GAPD
Not the approved symbol: PRR10
Not the approved symbol: TGIF2-C20orf24
Not the approved symbol: C5orf64
Not the approved symbol: MGC50722
Not the approved symbol: LOC392787
Not the approved symbol: HIST1H4L
Not the approved symbol: CCDC113
Not the approved symbol: NPEPPSL1
Not the approved symbol: FLJ11292
Not the approved symbo

,Input,Approved symbol
0,LOC554223,NaN
1,TRGV8,TRGV8
2,CEFIP,C10orf71
3,LINC01565,LINC01565
4,MUXA,NaN


Number of interactions: 759122


,Uniprot_A,Uniprot_B,PPI_pair,Gene_approved_A,Gene_approved_B,gene_pair
0,A0A024R0H7,Q8TB24,A0A024R0H7:Q8TB24,WDR77,RIN3,RIN3:WDR77
1,A0A024R0L6,Q5S007,A0A024R0L6:Q5S007,PAFAH1B3,LRRK2,LRRK2:PAFAH1B3
2,A0A024R1A3,Q5S007,A0A024R1A3:Q5S007,UBA1,LRRK2,LRRK2:UBA1
3,A0A024R2I8,O15379,A0A024R2I8:O15379,THRB,HDAC3,HDAC3:THRB
4,A0A024R2I8,O75376,A0A024R2I8:O75376,THRB,NCOR1,NCOR1:THRB


In [ ]:
## Merge and process BioPlex lists

# concatenate interactomes (downloaded from BioPlex)
bioplex_file_list = ["BioPlex_293T_Network_10K_Dec_2019.tsv", "BioPlex_HCT116_Network_5.5K_Dec_2019.tsv", "BioPlex_interactionList_v2.tsv", "BioPlex_interactionList_v4a.tsv"]
bioplex_interactomes = []
for bioplex_file in bioplex_file_list:
    bioplex_file_path = os.path.join(root_dir, "_DATA/_COMMON/network/BioPlex/", bioplex_file)
    interactome = pd.read_table(bioplex_file_path)
    interactome.columns = [x.replace(" ", "") for x in interactome.columns]
    interactome["gene_pair"] = interactome.apply(lambda row: ":".join(sorted([row["SymbolA"], row["SymbolB"]])), axis=1)
    interactome = interactome.drop_duplicates(subset=["gene_pair"])
    interactome = interactome[["UniprotA", "UniprotB", "SymbolA", "SymbolB", "gene_pair"]]
    # display(interactome.head())
    bioplex_interactomes.append(interactome)
bioplex_all_interactome = pd.concat(bioplex_interactomes, axis=0).drop_duplicates(subset=["gene_pair"]).reset_index(drop=True)

# !! Issue: but many original gene names in BioPlex are not the approved HGNC symbols, need to map them to the approved symbols

# # generate the gene lis for those genes that are not approved HGNC symbols
# hgnc_gene_info = pd.read_table(os.path.join(root_dir, "_DATA/_COMMON/genes/HGNC_proteincoding.txt"))
# bioplex_all_interactome_genes = pd.DataFrame(list(set(list(bioplex_all_interactome["SymbolA"].unique()) + list(bioplex_all_interactome["SymbolB"].unique()))))
# bioplex_all_interactome_genes[~bioplex_all_interactome_genes[0].isin(list(hgnc_gene_info["symbol"]))].to_csv(os.path.join(root_dir, "_DATA/_COMMON/network/BioPlex/BioPlex_all_genes_notapproved.txt"), sep='\t', index=None, header=None)

# load the appoved HGNC mapping only for those genes that are not approved symbols
bioplex_all_notapproved_genes_hgnc = pd.read_table(os.path.join(root_dir, "_DATA/_COMMON/network/BioPlex/BioPlex_all_genes_notapproved_hgnc.csv"), sep=',')
genes_approved = []
for input_gene in bioplex_all_notapproved_genes_hgnc["Input"].unique():
    hgnc_inputgene = bioplex_all_notapproved_genes_hgnc[bioplex_all_notapproved_genes_hgnc["Input"] == input_gene]
    hgnc_inputgene_appr = hgnc_inputgene[hgnc_inputgene["Match type"] == "Approved symbol"]
    if len(hgnc_inputgene_appr) == 0: # for input genes that are not approved HGNC symbols, replace with all the mapped approved symbols
        print("Not the approved symbol:", input_gene)
        # display(hgnc_inputgene)
        genes_approved.append(hgnc_inputgene[["Input", "Approved symbol"]])
    elif len(hgnc_inputgene_appr) == 1: # for input genes that are approved HGNC symbols
        genes_approved.append(hgnc_inputgene_appr[["Input", "Approved symbol"]])
    elif len(hgnc_inputgene_appr) > 1: # for input genes that are approved HGNC symbols
        # display(genes_hgnc_inputgene_apr)
        genes_approved.append(hgnc_inputgene_appr[["Input", "Approved symbol"]])
genes_approved = pd.concat(genes_approved, axis=0).reset_index(drop=True)
display(genes_approved.head())

# finally, using the clean, filled and mostly approved symbols as gene names to generate gene_pairs and filter out duplicated ones
bioplex_all_interactome = pd.merge(bioplex_all_interactome, genes_approved, left_on="SymbolA", right_on="Input", how="left").rename(columns={"Approved symbol":"SymbolA_approved"}).drop(columns=["Input"])
bioplex_all_interactome["SymbolA_approved"] = bioplex_all_interactome.apply(lambda row: row["SymbolA_approved"] if pd.isna(row["SymbolA_approved"]) == False else row["SymbolA"], axis=1) # if no approved symbol, just use the original one
bioplex_all_interactome = pd.merge(bioplex_all_interactome, genes_approved, left_on="SymbolB", right_on="Input", how="left").rename(columns={"Approved symbol":"SymbolB_approved"}).drop(columns=["Input"])
bioplex_all_interactome["SymbolB_approved"] = bioplex_all_interactome.apply(lambda row: row["SymbolB_approved"] if pd.isna(row["SymbolB_approved"]) == False else row["SymbolB"], axis=1) # if no approved symbol, just use the original one
bioplex_all_interactome["gene_pair"] = bioplex_all_interactome.apply(lambda row: ":".join(sorted([row["SymbolA_approved"], row["SymbolB_approved"]])), axis=1)
bioplex_all_interactome = bioplex_all_interactome.drop_duplicates(subset=["gene_pair"]).reset_index(drop=True)
bioplex_all_interactome = bioplex_all_interactome[["UniprotA", "UniprotB", "SymbolA_approved", "SymbolB_approved", "gene_pair"]]

# save the merged interactome with filled gene names and approved symbols, and the gene pairs with approved symbols
bioplex_all_interactome_file = os.path.join(root_dir, "_DATA/_COMMON/network/BioPlex/BioPlex_all.tsv")
bioplex_all_interactome.to_csv(bioplex_all_interactome_file, sep='\t', index=None)
print("Number of interactions:", len(bioplex_all_interactome))
bioplex_all_interactome.head()

Not the approved symbol: LOC554223
Not the approved symbol: NBPF24
Not the approved symbol: LOC100507685
Not the approved symbol: KIAA1161
Not the approved symbol: C19orf20
Not the approved symbol: RAB7L1
Not the approved symbol: C1orf212
Not the approved symbol: C20orf106
Not the approved symbol: PET112L
Not the approved symbol: FAM122C
Not the approved symbol: CXorf23
Not the approved symbol: GRINL1A
Not the approved symbol: C5orf35
Not the approved symbol: H2AFB1
Not the approved symbol: MYST1
Not the approved symbol: KIAA0748
Not the approved symbol: GATSL3
Not the approved symbol: RAD51L1
Not the approved symbol: C10orf84
Not the approved symbol: YARS
Not the approved symbol: C9orf139
Not the approved symbol: KIAA1009
Not the approved symbol: ARSE
Not the approved symbol: STRA13
Not the approved symbol: C1orf38
Not the approved symbol: FAM18A
Not the approved symbol: FAM166B
Not the approved symbol: C11orf10
Not the approved symbol: COPG
Not the approved symbol: FAM45A
Not the app

,Input,Approved symbol
0,LOC554223,NaN
1,NBPF24,NBPF11
2,LOC100507685,NaN
3,KIAA1161,MYORG
4,C19orf20,TPGS1


Number of interactions: 183022


,UniprotA,UniprotB,SymbolA_approved,SymbolB_approved,gene_pair
0,P00813,A5A3E0,ADA,POTEF,ADA:POTEF
1,Q8N7W2-2,P26373,BEND7,RPL13,BEND7:RPL13
2,Q8N7W2-2,Q09028-3,BEND7,RBBP4,BEND7:RBBP4
3,Q8N7W2-2,Q9Y3U8,BEND7,RPL36,BEND7:RPL36
4,Q8N7W2-2,P36578,BEND7,RPL4,BEND7:RPL4


In [ ]:
## Merge and process OpenCell interactome

opencell_interactome_file = os.path.join(root_dir, "_DATA/_COMMON/network/OpenCell/opencell-protein-interactions.csv")
opencell_interactome = pd.read_table(opencell_interactome_file, sep=',')
opencell_interactome["interactor_gene_name"] = opencell_interactome["interactor_gene_name"].apply(lambda x: x.split(";"))
opencell_interactome = opencell_interactome.explode("interactor_gene_name")
opencell_interactome = opencell_interactome[["target_gene_name", "interactor_gene_name"]].reset_index(drop=True)
opencell_interactome["gene_pair"] = opencell_interactome.progress_apply(lambda row: ":".join(sorted([row["target_gene_name"], row["interactor_gene_name"]])), axis=1)
opencell_interactome = opencell_interactome.drop_duplicates(subset=["gene_pair"]).reset_index(drop=True)

# !! Issue: but many original gene names in OpenCell are not the approved HGNC symbols, need to map them to the approved symbols

# # generate the gene lis for those genes that are not approved HGNC symbols
# hgnc_gene_info = pd.read_table(os.path.join(root_dir, "_DATA/_COMMON/genes/HGNC_proteincoding.txt"))
# opencell_interactome_genes = pd.DataFrame(list(set(list(opencell_interactome["target_gene_name"].unique()) + list(opencell_interactome["interactor_gene_name"].unique()))))
# opencell_interactome_genes[~opencell_interactome_genes[0].isin(list(hgnc_gene_info["symbol"]))].to_csv(os.path.join(root_dir, "_DATA/_COMMON/network/OpenCell/opencell_all_genes_notapproved.txt"), sep='\t', index=None, header=None)

# load the appoved HGNC mapping only for those genes that are not approved symbols
opencell_interactome_genes_hgnc = pd.read_table(os.path.join(root_dir, "_DATA/_COMMON/network/OpenCell/opencell_all_genes_notapproved_hgnc.csv"), sep=',')
genes_approved = []
for input_gene in opencell_interactome_genes_hgnc["Input"].unique():
    hgnc_inputgene = opencell_interactome_genes_hgnc[opencell_interactome_genes_hgnc["Input"] == input_gene]
    hgnc_inputgene_appr = hgnc_inputgene[hgnc_inputgene["Match type"] == "Approved symbol"]
    if len(hgnc_inputgene_appr) == 0: # for input genes that are not approved HGNC symbols, replace with all the mapped approved symbols
        print("Not the approved symbol:", input_gene)
        # display(hgnc_inputgene)
        genes_approved.append(hgnc_inputgene[["Input", "Approved symbol"]])
    elif len(hgnc_inputgene_appr) == 1:
        genes_approved.append(hgnc_inputgene_appr[["Input", "Approved symbol"]])
    elif len(hgnc_inputgene_appr) > 1:
        # display(genes_hgnc_inputgene_apr)
        genes_approved.append(hgnc_inputgene_appr[["Input", "Approved symbol"]])
genes_approved = pd.concat(genes_approved, axis=0).reset_index(drop=True)
display(genes_approved.head())

# finally, using the clean, filled and mostly approved symbols as gene names to generate gene_pairs and filter out duplicated ones
opencell_interactome = pd.merge(opencell_interactome, genes_approved, left_on="target_gene_name", right_on="Input", how="left").rename(columns={"Approved symbol":"target_gene_name_approved"}).drop(columns=["Input"])
opencell_interactome["target_gene_name_approved"] = opencell_interactome.apply(lambda row: row["target_gene_name_approved"] if pd.isna(row["target_gene_name_approved"]) == False else row["target_gene_name"], axis=1) # if no approved symbol, just use the original one
opencell_interactome = pd.merge(opencell_interactome, genes_approved, left_on="interactor_gene_name", right_on="Input", how="left").rename(columns={"Approved symbol":"interactor_gene_name_approved"}).drop(columns=["Input"])
opencell_interactome["interactor_gene_name_approved"] = opencell_interactome.apply(lambda row: row["interactor_gene_name_approved"] if pd.isna(row["interactor_gene_name_approved"]) == False else row["interactor_gene_name"], axis=1) # if no approved symbol, just use the original one
opencell_interactome["gene_pair"] = opencell_interactome.apply(lambda row: ":".join(sorted([row["target_gene_name_approved"], row["interactor_gene_name_approved"]])), axis=1)
opencell_interactome = opencell_interactome.drop_duplicates(subset=["gene_pair"]).reset_index(drop=True)
opencell_interactome = opencell_interactome[["target_gene_name_approved", "interactor_gene_name_approved", "gene_pair"]]

# save the merged interactome with filled gene names and approved symbols, and the gene pairs with approved symbols
opencell_all_interactome_file = os.path.join(root_dir, "_DATA/_COMMON/network/OpenCell/opencell_all.csv")
opencell_interactome.to_csv(opencell_all_interactome_file, sep=",", index=None)
print("Number of interactions:", len(opencell_interactome))
opencell_interactome.head()

100%|██████████| 31838/31838 [00:00<00:00, 189798.79it/s]


Not the approved symbol: FLJ00186
Not the approved symbol: DKFZP434D199
Not the approved symbol: SEPTININININ3
Not the approved symbol: FAM45A
Not the approved symbol: TMEM2
Not the approved symbol: BAT3
Not the approved symbol: SEPTININININ5
Not the approved symbol: MRPS36
Not the approved symbol: TOMM20A
Not the approved symbol: KIAA1211L
Not the approved symbol: BCAR2
Not the approved symbol: C17ORF70
Not the approved symbol: CDC2
Not the approved symbol: SKIV2L2
Not the approved symbol: KIAA1468
Not the approved symbol: NPEPPSL1
Not the approved symbol: HIST2H2AA3
Not the approved symbol: TDP43
Not the approved symbol: C3ORF58
Not the approved symbol: WIBG
Not the approved symbol: WAPAL
Not the approved symbol: HIST1H2AB
Not the approved symbol: C7ORF50
Not the approved symbol: DARS
Not the approved symbol: ATP5B
Not the approved symbol: KIAA0368
Not the approved symbol: GBA
Not the approved symbol: HIST1H2BK
Not the approved symbol: MLK4
Not the approved symbol: C7ORF26
Not the ap

,Input,Approved symbol
0,FLJ00186,NaN
1,PMF1-BGLAP,PMF1-BGLAP
2,DKFZP434D199,NaN
3,DDX12P,DDX12P
4,AKAP2,AKAP2


Number of interactions: 30096


,target_gene_name_approved,interactor_gene_name_approved,gene_pair
0,AAMP,ARGLU1,AAMP:ARGLU1
1,AAMP,CWF19L2,AAMP:CWF19L2
2,AAMP,PRPF40A,AAMP:PRPF40A
3,AAMP,RPL10,AAMP:RPL10
4,AAMP,RSRC1,AAMP:RSRC1


In [ ]:
# Merge all literature interactomes (this should be the "literature-PPI" list used in the main text, which is the union of all literature PPIs from HINT, BioPlex and OpenCell, with gene pairs using the approved HGNC symbols)

HINT_all_interactome = pd.read_table(os.path.join(root_dir, "_DATA/_COMMON/network/HINT/HomoSapiens_both_all.txt"))[["Gene_approved_A", "Gene_approved_B", "gene_pair"]].rename(columns={"Gene_approved_A":"gene_A", "Gene_approved_B":"gene_B"})
bioplex_all_interactome = pd.read_table(os.path.join(root_dir, "_DATA/_COMMON/network/BioPlex/BioPlex_all.tsv"))[["SymbolA_approved", "SymbolB_approved", "gene_pair"]].rename(columns={"SymbolA_approved":"gene_A", "SymbolB_approved":"gene_B"})
opencell_interactome = pd.read_table(os.path.join(root_dir, "_DATA/_COMMON/network/OpenCell/opencell_all.csv"), sep=',')[["target_gene_name_approved", "interactor_gene_name_approved", "gene_pair"]].rename(columns={"target_gene_name_approved":"gene_A", "interactor_gene_name_approved":"gene_B"})

# !! Issue: many genes might be the previous symbols and not the approved HGNC symbols
literature_all_interactome = pd.concat([HINT_all_interactome, bioplex_all_interactome, opencell_interactome], axis=0).drop_duplicates(subset=["gene_pair"]).reset_index(drop=True)
literature_all_interactome.to_csv(os.path.join(root_dir, "_DATA/_COMMON/network/all_literature/literature_all.txt"), sep='\t', index=None)
print("Number of interactions:", len(literature_all_interactome))
literature_all_interactome.head()

Number of interactions: 765276


,gene_A,gene_B,gene_pair
0,WDR77,RIN3,RIN3:WDR77
1,PAFAH1B3,LRRK2,LRRK2:PAFAH1B3
2,UBA1,LRRK2,LRRK2:UBA1
3,THRB,HDAC3,HDAC3:THRB
4,THRB,NCOR1,NCOR1:THRB
